<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/04-spark-sql/03-hive-metastore-basics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 4.3 — Hive metastore basics

This notebook writes real tables. They land in a `spark-warehouse/`
folder **next to this notebook** — the last cell cleans up. (We use the
default in-memory catalog, not `enableHiveSupport()`, to avoid local
Derby sharp edges — see the lesson. Windows note: `saveAsTable` needs
the lesson-1.2 winutils setup.)

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("4.3-metastore")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

print("warehouse dir:", spark.conf.get("spark.sql.warehouse.dir"))

## A managed table: saveAsTable, read back by name

In [ ]:
orders.write.mode("overwrite").saveAsTable("orders_tbl")

spark.table("orders_tbl").count()

In [ ]:
# It's a real table, not a temp view — see the catalog flags (lesson 4.2):
for t in spark.catalog.listTables():
    print(f"{t.name:<12} type={t.tableType:<10} isTemporary={t.isTemporary}")

# ...and real FILES on disk (this is what a temp view never had):
import pathlib
wh = pathlib.Path("spark-warehouse/orders_tbl")
print("\nfiles:", [p.name for p in wh.iterdir()] if wh.exists() else "—")
print("format: parquet (the default — Module 5)")

## DESCRIBE EXTENDED — where managed-vs-external shows itself

In [ ]:
spark.sql("DESCRIBE EXTENDED orders_tbl") \
    .filter(col("col_name").isin("Type", "Location", "Provider")) \
    .show(truncate=False)

## Modes, CTAS, and appending

In [ ]:
# CTAS — the SQL spelling of saveAsTable
spark.sql("""
    CREATE TABLE in_orders AS
    SELECT * FROM orders_tbl WHERE country = 'IN'
""")
print("in_orders:", spark.table("in_orders").count())

# append mode adds rows
spark.table("in_orders").limit(2).write.mode("append").saveAsTable("in_orders")
print("after append:", spark.table("in_orders").count())

## DROP TABLE on a managed table deletes the DATA — watch

In [ ]:
print("files exist before drop:", wh.exists())
spark.sql("DROP TABLE orders_tbl")
print("catalog entry gone:     ", not spark.catalog.tableExists("orders_tbl"))
print("files exist after drop: ", wh.exists(), "  <- managed = data deleted too")

An **external** table (`.option("path", ...)` before `saveAsTable`) would
have kept its files through that drop — that's the entire
managed-vs-external distinction, and why data lakes default to external.

## Cleanup

In [ ]:
spark.sql("DROP TABLE IF EXISTS in_orders")

import shutil
shutil.rmtree("spark-warehouse", ignore_errors=True)   # remove leftover warehouse dir
print("cleaned up.")

## Exercises

1. Create `orders_ext` as an EXTERNAL table (write `option("path", "./ext_orders_data")` first), drop it, and verify the files survived. Clean up the folder manually afterwards.
2. What does `mode("overwrite").saveAsTable` do to an external table's files? Reason from the ownership model, then test.
3. Restart the kernel (in-memory catalog!) and check `spark.catalog.tableExists("in_orders")` before the cleanup cell has re-run — what survived the restart, the entry or the files? Reconcile with the lesson's 'orphaned files' warning.
4. (Optional, sharp edges) Re-run this notebook with `.enableHiveSupport()` in the builder. What new directory appears next to the notebook? Restart and confirm tables now survive. Delete `metastore_db/` when done.

In [ ]:
# your exercise code here
